In [18]:
import os
from openai import OpenAI

client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))
print("✓ Client started successfully")

✓ Client started successfully


In [12]:
client.chat.completions.create(
    model="gpt-4o",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "What is the capital of France?"}
    ] 
)

ChatCompletion(id='chatcmpl-DY7vyU4ePjRlIib9jgRIQsXFiiOaX', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='The capital of France is Paris.', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None))], created=1777026022, model='gpt-4o-2024-08-06', object='chat.completion', service_tier='default', system_fingerprint='fp_054fa161b4', usage=CompletionUsage(completion_tokens=7, prompt_tokens=24, total_tokens=31, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0)))

In [13]:
# Install: pip install datasets
from datasets import load_dataset
import requests
from PIL import Image
import pandas as pd
from pathlib import Path
 
# Load dataset from HuggingFace
print("Loading product dataset...")
try:
    # Try loading the dataset
    dataset = load_dataset("ashraq/fashion-product-images-small", split="train[:100]")  # First 100 samples
    print(f"✓ Loaded {len(dataset)} products")
    
    # Convert to pandas for easier manipulation
    products_df = pd.DataFrame(dataset)
    print(f"Dataset columns: {products_df.columns.tolist()}")
    
except Exception as e:
    print(f"⚠ Could not load HuggingFace dataset: {e}")
    print("Using local images instead...")
    
    # Alternative: Use local images
    # Create a products.json file with product information
    products_data = [
        {
            "id": 1,
            "name": "Wireless Headphones",
            "price": 79.99,
            "category": "Electronics",
            "image_path": "images/product1.jpg"
        },
        # Add more products...
    ]
    
    products_df = pd.DataFrame(products_data)
 
# Create images directory
images_dir = Path("product_images")
images_dir.mkdir(exist_ok=True)
 
print(f"\n✓ Dataset prepared!")
print(f"  Total products: {len(products_df)}")

Loading product dataset...
✓ Loaded 100 products
Dataset columns: ['id', 'gender', 'masterCategory', 'subCategory', 'articleType', 'baseColour', 'season', 'year', 'usage', 'productDisplayName', 'image']

✓ Dataset prepared!
  Total products: 100


In [14]:
import base64
from io import BytesIO

def encode_image_to_base64(image):
    """Convert a PIL Image to base64 string"""
    buffer = BytesIO()
    image.save(buffer, format="JPEG")
    buffer.seek(0)
    img_bytes = buffer.read()
    return base64.b64encode(img_bytes).decode('utf-8')


first_image = dataset[0]['image']
encoded = encode_image_to_base64(first_image)

print(f"✓ Image encoded successfully")
print(f"  Original type: {type(first_image)}")
print(f"  Base64 length: {len(encoded)} characters")
print(f"  First 50 chars: {encoded[:50]}...")

✓ Image encoded successfully
  Original type: <class 'PIL.JpegImagePlugin.JpegImageFile'>
  Base64 length: 2388 characters
  First 50 chars: /9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAAgGBgcGBQgHBwcJCQ...


In [15]:
def create_product_listing_prompt(product_name, price, category, additional_info=None):
    """
    Create a prompt for generating product listings.
    
    Parameters:
    - product_name: Name of the product
    - price: Price of the product
    - category: Product category
    - additional_info: Optional additional information
    
    Returns:
    - Formatted prompt string
    """
    prompt = f"""You are an expert e-commerce copywriter. Analyze the product image and create a compelling product listing.
 
Product Information:
- Name: {product_name}
- Price: ${price:.2f}
- Category: {category}
{f'- Additional Info: {additional_info}' if additional_info else ''}
 
Please create a professional product listing that includes:
 
1. **Product Title** (catchy, SEO-friendly, 60 characters max)
2. **Product Description** (detailed, 150-200 words)
   - Highlight key features and benefits
   - Use persuasive language
   - Include relevant details visible in the image
3. **Key Features** (bullet points, 5-7 items)
4. **SEO Keywords** (comma-separated, 10-15 relevant keywords)
 
Format your response as JSON with the following structure:
{{
    "title": "Product title here",
    "description": "Full description here",
    "features": ["Feature 1", "Feature 2", ...],
    "keywords": "keyword1, keyword2, ..."
}}
 
Be specific about what you see in the image. Mention colors, materials, design elements, and any distinctive features."""
    
    return prompt
 
# Test prompt creation
test_prompt = create_product_listing_prompt(
    product_name="Wireless Bluetooth Headphones",
    price=79.99,
    category="Electronics",
    additional_info="Noise cancelling, 30-hour battery"
)
 
print("\n" + "="*50)
print("PROMPT TEMPLATE")
print("="*50)
print(test_prompt[:500] + "...")  # Show first 500 characters


PROMPT TEMPLATE
You are an expert e-commerce copywriter. Analyze the product image and create a compelling product listing.

Product Information:
- Name: Wireless Bluetooth Headphones
- Price: $79.99
- Category: Electronics
- Additional Info: Noise cancelling, 30-hour battery

Please create a professional product listing that includes:

1. **Product Title** (catchy, SEO-friendly, 60 characters max)
2. **Product Description** (detailed, 150-200 words)
   - Highlight key features and benefits
   - Use persuasive ...


In [16]:
import json
import re

def analyze_product(image, product_info):
    """Analyze a product image using GPT-4o"""
    
    # Step 1: Prepare image
    encoded_image = encode_image_to_base64(image)
    
    # Step 2: Call the API
    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {
                "role": "user",
                "content": [
                    {
                        "type": "image_url",
                        "image_url": {
                            "url": f"data:image/jpeg;base64,{encoded_image}"
                        }
                    },
                    {
                        "type": "text",
                        "text": "Analyze this product image and return a JSON with: {\"color\": \"main color\", \"style\": \"product style\", \"tags\": [\"tag1\", \"tag2\", \"tag3\"]}. Return only the JSON, nothing else."
                    }
                ]
            }
        ]
    )
    
    # Step 3: Handle response
    raw_output = response.choices[0].message.content
    
    # Step 4: Parse JSON
    print("Raw output:", repr(raw_output))
    clean = re.sub(r'```json\s*|\s*```', '', raw_output).strip()
    parsed = json.loads(clean)
    
    return parsed

# Test con el primer producto
first_product = dataset[0]
result = analyze_product(first_product['image'], first_product)

print("✓ API call successful")
print(json.dumps(result, indent=2))

Raw output: '```json\n{"color": "blue", "style": "casual", "tags": ["plaid", "shirt", "short sleeves"]}\n```'
✓ API call successful
{
  "color": "blue",
  "style": "casual",
  "tags": [
    "plaid",
    "shirt",
    "short sleeves"
  ]
}


In [17]:
import time

results = []
errors = []

for i in range(5):
    try:
        product = dataset[i]
        print(f"Processing product {i+1}/5...")
        result = analyze_product(product['image'], product)
        result['product_id'] = i
        results.append(result)
        time.sleep(1)
    except Exception as e:
        print(f"Error in product {i+1}: {e}")
        errors.append({"product_id": i, "error": str(e)})

with open("results.json", "w") as f:
    json.dump(results, f, indent=2)

print(f"\n✓ Processed: {len(results)} products")
print(f"✓ Errors: {len(errors)}")
print(f"✓ Saved to results.json")

Processing product 1/5...
Raw output: '```json\n{\n  "color": "blue",\n  "style": "checkered shirt",\n  "tags": ["plaid", "casual", "short sleeves"]\n}\n```'
Processing product 2/5...
Raw output: '```json\n{\n  "color": "blue",\n  "style": "casual",\n  "tags": ["jeans", "denim", "men\'s fashion"]\n}\n```'
Processing product 3/5...
Raw output: '{"color": "silver", "style": "elegant", "tags": ["watch", "jewelry", "accessory"]}'
Processing product 4/5...
Raw output: '{"color": "black", "style": "track pants", "tags": ["sportswear", "casual", "athletic"]}'
Processing product 5/5...
Raw output: '{"color": "grey", "style": "polo shirt", "tags": ["casual", "short sleeves", "men\'s fashion"]}'

✓ Processed: 5 products
✓ Errors: 0
✓ Saved to results.json
